# EEG Exploratory Data Analysis

This notebook consumes Person 1 compressed processed archives from `data/processed/`. It documents amplitude and variability, the five requested frequency bands, channel differences, channel correlations, recorded-condition comparisons, and the EDA-informed feature table.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd()
if not (repo_root / 'src').is_dir():
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root))

from src.eda.channel_analysis import channel_difference, channel_summary
from src.eda.correlation_condition_analysis import (
    channel_correlation, condition_summary, resting_vs_cognitive,
)
from src.eda.frequency_domain import BANDS, band_powers
from src.eda.time_domain import load_processed_archive, summarize_time_domain
from src.feature_selection.select_features import select_features

figure_dir = repo_root / 'results' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
archives = sorted((repo_root / 'data' / 'processed').glob('*.npz'))
if not archives:
    raise FileNotFoundError('No Person 1 .npz archives found in data/processed/')

data_parts = []
metadata = []
channel_names = None
sampling_frequency = None
for archive in archives:
    archive_data, archive_metadata = load_processed_archive(archive)
    if not archive_metadata:
        continue
    archive_channels = archive_metadata[0]['channel_names']
    archive_frequency = float(archive_metadata[0]['sampling_frequency'])
    if channel_names is None:
        channel_names = archive_channels
        sampling_frequency = archive_frequency
    if any(item.get('channel_names') != archive_channels for item in archive_metadata):
        raise ValueError(f'Channel order differs within {archive.name}')
    if any(float(item.get('sampling_frequency', archive_frequency)) != archive_frequency for item in archive_metadata):
        raise ValueError(f'Sampling frequency differs within {archive.name}')
    if archive_channels != channel_names:
        raise ValueError(f'Channel order differs between processed archives: {archive.name}')
    if archive_frequency != sampling_frequency:
        raise ValueError(f'Sampling frequency differs between processed archives: {archive.name}')
    data_parts.append(archive_data)
    metadata.extend(archive_metadata)
if not data_parts:
    raise ValueError('No clean segments remain after anomaly exclusion')
data = np.concatenate(data_parts, axis=0)
print(f'Loaded {len(archives)} archive(s): {data.shape}, {sampling_frequency} Hz, {len(channel_names)} channels')

## Time domain: amplitude, spread, and variability

In [ ]:
time_summary = summarize_time_domain(data, channel_names)
mean_std_by_channel = time_summary['std'].mean(axis=0)
top_channels = np.argsort(mean_std_by_channel)[-10:][::-1]
plt.figure(figsize=(10, 4))
plt.bar([channel_names[i] for i in top_channels], mean_std_by_channel[top_channels])
plt.ylabel('Mean standard deviation (V)')
plt.title('Highest EEG channel variability')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(figure_dir / 'time_domain_channel_variability.png', dpi=160)
plt.show()

## Frequency domain: Delta, Theta, Alpha, Beta, and Gamma power

In [ ]:
powers, band_names = band_powers(data, sampling_frequency, relative=True)
mean_band_power = powers.mean(axis=(0, 1))
plt.figure(figsize=(8, 4))
plt.bar([name.title() for name in band_names], mean_band_power)
plt.ylabel('Relative power')
plt.title('Mean relative EEG band power')
plt.tight_layout()
plt.savefig(figure_dir / 'frequency_band_power.png', dpi=160)
plt.show()

## Channels and conditions

In [ ]:
channel_rows = channel_summary(data, channel_names)
condition_rows = condition_summary(data, metadata)
condition_names = [row['condition'] for row in condition_rows]
condition_amplitudes = [row['mean_absolute_amplitude'] for row in condition_rows]
plt.figure(figsize=(8, 4))
plt.bar(condition_names, condition_amplitudes)
plt.ylabel('Mean absolute amplitude (V)')
plt.title('Amplitude by recorded condition')
plt.tight_layout()
plt.savefig(figure_dir / 'condition_amplitude.png', dpi=160)
plt.show()

correlation = channel_correlation(data, channel_names)
plt.figure(figsize=(7, 6))
plt.imshow(correlation, vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar(label='Pearson correlation')
plt.title('EEG channel correlation')
plt.tight_layout()
plt.savefig(figure_dir / 'channel_correlation.png', dpi=160)
plt.show()

print('Resting versus cognitive:', resting_vs_cognitive(data, metadata))

In [ ]:
first, second = channel_names[:2]
difference = channel_difference(data, channel_names, first, second)
plt.figure(figsize=(10, 3))
plt.plot(difference['difference'][0])
plt.xlabel('Sample')
plt.ylabel('Voltage difference (V)')
plt.title(f'{first} minus {second}, first clean segment')
plt.tight_layout()
plt.savefig(figure_dir / 'channel_difference.png', dpi=160)
plt.show()

## Finalized EDA-informed feature set

The selector begins with per-channel standard deviation, RMS, mean absolute amplitude, and relative band power. It ranks features by condition separation when multiple conditions are present, falls back to variance otherwise, and returns a bounded table for downstream modelling.

In [ ]:
selected = select_features(
    data, metadata, sampling_frequency, channel_names, max_features=30
)
print('Final feature count:', len(selected.feature_names))
print('Selected features:', selected.feature_names)
print('Feature matrix shape:', selected.values.shape)